In [ ]:
import os
import glob
import random
import pandas as pd
from Bio import PDB
from Bio.PDB.cealign import CEAligner

from tqdm.notebook import tqdm

# Config
TARGET_PDB = "../inputs/rosetta/carA_ref/6OZ1.pdb"

INPUT_DIR = "structure_single"
OUTPUT_DIR = "../inputs/rosetta/carA_single_target_holo_adi_amp_swiss/"

LIGAND_MOL2 = [
    ("../inputs/rosetta/subs/adi.mol2", "ADI"),
    ("../inputs/rosetta/subs/amp.mol2", "AMP")
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
parser = PDB.PDBParser(QUIET=True)
io = PDB.PDBIO()


def parse_mol2_to_hetatm(mol2_path: str, res_name: str = "LIG", chain_id: str = "Z") -> list:
    """Parse mol2 @<TRIPOS>ATOM section and convert to PDB HETATM format lines."""
    hetatm_lines = []
    in_atom_block = False

    with open(mol2_path, "r") as f:
        for line in f:
            if line.startswith("@<TRIPOS>ATOM"):
                in_atom_block = True
                continue
            if line.startswith("@<TRIPOS>") and in_atom_block:
                break
            if not in_atom_block or not line.strip():
                continue

            parts = line.split()
            atom_serial = int(parts[0])
            atom_name = parts[1][:4].ljust(4)
            x, y, z = float(parts[2]), float(parts[3]), float(parts[4])
            element = parts[5].split(".")[0][:2].upper()

            hetatm_line = (
                f"HETATM{atom_serial:5d} {atom_name} {res_name:3s} {chain_id}"
                f"{1:4d}    {x:8.3f}{y:8.3f}{z:8.3f}{1.00:6.2f}{0.00:6.2f}          {element:>2s}\n"
            )
            hetatm_lines.append(hetatm_line)

    print(f"Ligand parsed: {len(hetatm_lines)} atoms from {os.path.basename(mol2_path)}")
    return hetatm_lines


def append_ligand_to_pdb(pdb_path: str, hetatm_lines: list):
    """Append ligand HETATM lines to an existing PDB file."""
    with open(pdb_path, "r") as f:
        lines = f.readlines()
    lines = [l for l in lines if not l.startswith("END")]
    with open(pdb_path, "w") as f:
        f.writelines(lines)
        f.writelines(hetatm_lines)
        f.write("END\n")


def align(mobile_path: str, target_struct, save_path: str):
    mobile_struct = parser.get_structure("mobile", mobile_path)

    aligner = CEAligner()
    aligner.set_reference(target_struct)
    aligner.align(mobile_struct)

    io.set_structure(mobile_struct)
    io.save(save_path)
    return round(float(aligner.rms), 3)


# Parse ligand(s)
if LIGAND_MOL2 is None:
    lig_paths = []
elif isinstance(LIGAND_MOL2, str):
    lig_paths = [LIGAND_MOL2]
else:
    lig_paths = list(LIGAND_MOL2)

chain_ids = "ZYXWVUT"
hetatm_lines = []
for i, (lig_path, lig_name) in enumerate(lig_paths):
    hetatm_lines.extend(parse_mol2_to_hetatm(lig_path, res_name=lig_name, chain_id=chain_ids[i]))

# Target defines the frame the ligand coordinates live in; mobiles are aligned onto it.
if TARGET_PDB is None:
    candidates = sorted(glob.glob(f"{INPUT_DIR}/*.pdb"))
    if not candidates:
        raise FileNotFoundError(f"No .pdb files found in {INPUT_DIR}")
    TARGET_PDB = random.choice(candidates)
    print(f"Target not specified — randomly picked from {INPUT_DIR}: {TARGET_PDB}")

target_struct = parser.get_structure("target", TARGET_PDB)
print(f"Target loaded as-is: {TARGET_PDB}\n")

# Main loop
results = []
for pdb_path in tqdm(sorted(glob.glob(f"{INPUT_DIR}/*.pdb"))):
    filename = os.path.basename(pdb_path)
    save_path = os.path.join(OUTPUT_DIR, filename)

    try:
        rmsd = align(pdb_path, target_struct, save_path)
        if hetatm_lines:
            append_ligand_to_pdb(save_path, hetatm_lines)
        print(f"[SUCCESS]    {filename}  RMSD = {rmsd} Å")
        results.append({"file": filename, "rmsd": rmsd})
    except Exception as e:
        print(f"[FAIL]  {filename} — {e}")

# Summary
summary = pd.DataFrame(results).sort_values("rmsd")
summary.to_csv("alignment_summary.csv", index=False)
print(f"\nAligned  : {len(results)} structures")
print(f"Mean RMSD: {summary['rmsd'].mean():.3f} Å")

Ligand parsed: 10 atoms from adi.mol2
Ligand parsed: 23 atoms from amp.mol2
Target loaded as-is: ../inputs/rosetta/carA_ref/6OZ1.pdb



  0%|          | 0/2 [00:00<?, ?it/s]

[SUCCESS]    A0A6G9XT36_swiss.pdb  RMSD = 0.703 Å
[SUCCESS]    A0AB38D5A4_swiss.pdb  RMSD = 0.06 Å

Aligned  : 2 structures
Mean RMSD: 0.381 Å
